# 缺失值重复值与清洗

学习目标：识别缺失与重复记录，按列制定清洗规则，并检查补值、筛选和去重对标签与统计的影响。

前置知识：类型转换、可空布尔、条件筛选、索引与列。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例均为自制小表，后续单元沿用 pd 和 np。清洗规则是每个例子的显式约定，不是所有数据都适用的默认处理。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按列补齐缺失信息

三条设备登记中，有一个设备名称和一个备注缺失。本例规定：名称是必填项，缺失名称的记录暂不进入结果；备注可以为空，展示时补成“未填写”。

dropna(subset=...) 只检查指定的必填列；fillna 用非缺失值替换缺失位置。先保留原表，分别观察筛选和补值的结果。

In [1]:
import pandas as pd
import numpy as np

raw = pd.DataFrame({"device": ["A", None, "C"], "note": [None, "待复核", "正常"]},
                   index=["r1", "r2", "r3"])
valid = raw.dropna(subset=["device"])
clean = valid.fillna({"note": "未填写"})
print(clean)  # 保留 r1、r3；r1 的 note 为未填写。
print(raw.shape, clean.shape)  # (3, 2)、(2, 2)。
print(raw.isna().sum())  # 原表 device、note 各有一个缺失，未被改写。
print(clean.dtypes)  # 两列均为 pandas 3 的 str。

   device note
r1      A  未填写
r3      C   正常
(3, 2) (2, 2)
device    1
note      1
dtype: int64
device    str
note      str
dtype: object


## 2 缺失标记与 dtype

pandas 按列类型使用不同缺失标记。外观不同并不代表只能分别检查；isna 和 notna 分别判断缺失与非缺失，保持原形状和标签。

| 名称 | 中文名称／含义 | 本章观察场合 |
| --- | --- | --- |
| None | Python 空值对象 | 输入或 object 列中的缺失 |
| np.nan | 浮点非数值标记 | float64 及 pandas 3 默认 str 的缺失 |
| pd.NaT | 日期时间缺失标记 | 日期时间与时间差 |
| pd.NA | pandas 缺失标记 | 可空 Int64、boolean、显式 string 等扩展类型 |

默认 str 与显式 string 的缺失语义不同；本课程安装了 PyArrow，两种字符串的底层存储均可使用它，但不能因此混淆缺失规则。

In [2]:
examples = {
    "object": pd.Series(["A", None], dtype=object),
    "float64": pd.Series([1.0, np.nan], dtype="float64"),
    "datetime64[us]": pd.Series([pd.Timestamp("2026-09-01"), pd.NaT], dtype="datetime64[us]"),
    "Int64": pd.Series([1, pd.NA], dtype="Int64"),
    "str": pd.Series(["A", None], dtype="str"),
    "string": pd.Series(["A", None], dtype="string"),
}
for name, values in examples.items():
    print(name, values.tolist(), values.isna().tolist())
    # 所有示例的第二项均被 isna 标为 True，显示的缺失标记依 dtype 而异。

object ['A', None] [False, True]
float64 [1.0, nan] [False, True]
datetime64[us] [Timestamp('2026-09-01 00:00:00'), NaT] [False, True]
Int64 [1, <NA>] [False, True]
str ['A', nan] [False, True]
string ['A', <NA>] [False, True]


不要用“等于缺失标记”代替 isna。NaN 与自身比较不相等，pd.NA 的相等比较传播未知结果；pd.NA 也不能直接用于 if 的真假判断。

In [3]:
print(np.nan == np.nan)  # False
print(pd.NA == pd.NA)  # <NA>：比较结果仍未知。
print(pd.isna([None, np.nan, pd.NaT, pd.NA]))  # 全部为 True。

False
<NA>
[ True  True  True  True]


In [4]:
# 预期 TypeError：pd.NA 的真值未知，bool 不能决定它是真还是假。
bool(pd.NA)

TypeError: boolean value of NA is ambiguous

## 3 空字符串与无穷值

空字符串、仅有空格的字符串和正负无穷不是 isna 默认认定的缺失。如果任务把它们视为无效，必须另外写出规则。

下面分别检查文本列和数值列。本例约定空文本与无限读数需要补充调查，再把它们转换为各自类型的缺失值；没有直接填成 0。

In [5]:
text = pd.Series(["A", "", " ", None], dtype="string")
readings = pd.Series([1.0, np.inf, -np.inf, np.nan], dtype="float64")
print(text.isna().tolist())  # [False, False, False, True]。
print(readings.isna().tolist())  # [False, False, False, True]。
normalized_text = text.replace({"": pd.NA, " ": pd.NA})
normalized_readings = readings.replace([np.inf, -np.inf], np.nan)
print(normalized_text.isna().tolist())  # [False, True, True, True]。
print(normalized_readings.notna().tolist())  # [True, False, False, False]。

[False, False, False, True]
[False, False, False, True]
[False, True, True, True]
[True, False, False, False]


replace 根据指定值或规则替换，fillna 只针对已经认定的缺失位置。按列传入替换映射可以避免误改其他字段。

下面约定 temperature 中 -999 表示设备未返回读数，但 count 中 -999 暂时保留供单独调查。是否应当替换由字段定义决定，不能全表统一猜测。

In [6]:
raw = pd.DataFrame({"temperature": [20.0, -999.0], "count": [3, -999]})
replaced = raw.replace({"temperature": {-999.0: np.nan}})
print(replaced)  # 只有 temperature 第二项变为 NaN。
print(replaced.dtypes)  # temperature 为 float64，count 为 int64。
print(raw.loc[1, "temperature"])  # -999.0：原表保留。

   temperature  count
0         20.0      3
1          NaN   -999
temperature    float64
count            int64
dtype: object
-999.0


## 4 删除哪些缺失记录

dropna 默认沿行删除，只要任一列缺失就删除该行。how='all' 只删除全部缺失的行；thresh 指定至少需要多少个非缺失值，它不能与 how 同时指定。

axis=1 改为删除列。不同参数表达不同的数据保留规则，不能只根据最后剩多少行来选择。

In [7]:
table = pd.DataFrame({"a": [1.0, np.nan, np.nan], "b": [2.0, 3.0, np.nan]},
                     index=["complete", "partial", "empty"])
print(table.dropna().index.tolist())  # ['complete']。
print(table.dropna(how="all").index.tolist())  # ['complete', 'partial']。
print(table.dropna(thresh=1).index.tolist())  # 至少一项非缺失，同样保留前两行。
print(table.dropna(subset=["a"]).index.tolist())  # 只要求 a，保留 complete。
print(table.dropna(axis=1).shape)  # (3, 0)：两列都含缺失，全部被删。

['complete']
['complete', 'partial']
['complete', 'partial']
['complete']
(3, 0)


## 5 填充值与列类型

填充值应与列类型和含义兼容。可空整数 Int64 可以用整数补齐，并保留类型；用 0 补值必须有业务依据，不能把未知数量自动当成零。

本例任务明确规定“没有新增数量就是 0”，备注缺失则写成“待录入”。实际测量缺失若表示未知，应保留缺失或单独记录，不能照搬该规则。

In [8]:
table = pd.DataFrame({"added": pd.array([2, None, 4], dtype="Int64"),
                      "note": pd.array(["正常", None, "复核"], dtype="string")})
filled = table.fillna({"added": 0, "note": "待录入"})
print(filled)  # 中间一行补成 0、待录入。
print(filled.dtypes)  # added 保持 Int64，note 保持 string。
print(table.isna().sum())  # 两列原本各缺失一项。

   added note
0      2   正常
1      0  待录入
2      4   复核
added     Int64
note     string
dtype: object
added    1
note     1
dtype: int64


## 6 按顺序传播数值

ffill 使用前一个有效值向后填充，bfill 使用后一个有效值向前填充。二者沿当前行顺序处理，不会自动判断记录属于哪个设备或是否已按时间排序。

下面是同一设备按时间先后排列的状态读数，limit=1 限制每段连续缺失最多传播一步。ffill 无法补最前面的缺失，bfill 无法补最后面的缺失。需要避免使用未来信息时，不应把 bfill 当作已知的历史观测。

In [9]:
values = pd.Series([np.nan, 10.0, np.nan, np.nan, 40.0, np.nan])
print(values.ffill(limit=1).tolist())  # [nan, 10, 10, nan, 40, 40]。
print(values.bfill(limit=1).tolist())  # [10, 10, nan, 40, 40, nan]。
print(values.tolist())  # 原序列保留缺失位置。

[nan, 10.0, 10.0, nan, 40.0, 40.0]
[10.0, 10.0, nan, 40.0, 40.0, nan]
[nan, 10.0, nan, nan, 40.0, nan]


## 7 按标签补齐另一张表

first.combine_first(second) 返回新表：优先保留 first 的非缺失值，只在它缺失的位置采用 second 的值。结果行列标签取双方并集，因此可能新增行和列。

下面原表顺序为 B、A，补充表为 C、B、A。核对 B 的缺失值按标签补为 20，A 已有的 10 优先于补充值 99；C 行和 note 列进入结果。

In [10]:
first = pd.DataFrame({"reading": [np.nan, 10.0]}, index=["B", "A"])
second = pd.DataFrame({"reading": [30.0, 20.0, 99.0], "note": ["新建", "补录", "备查"]},
                      index=["C", "B", "A"])
combined = first.combine_first(second)
print(combined)  # A=10、B=20、C=30；note 也进入结果。
print(combined.index.tolist(), combined.columns.tolist(), combined.shape)
# 当前结果行标签 A、B、C；列标签 reading、note；形状 (3, 2)。
print(first)  # 原表 B 的 reading 仍缺失。
print(combined.dtypes)  # reading 为 float64，note 为 str。

   reading note
A     10.0   备查
B     20.0   补录
C     30.0   新建
['A', 'B', 'C'] ['reading', 'note'] (3, 2)
   reading
B      NaN
A     10.0
reading    float64
note           str
dtype: object


combine_first 与 update 都按标签匹配，但用途不同：前者返回可扩展标签的新结果，后者修改调用对象并限制在原表范围内。update 默认还会覆盖原表已有值；只想补缺失时须用 overwrite=False。

继续用同一份 first、second 比较，显式复制 first 后再更新。

In [11]:
updated = first.copy()
updated.update(second, overwrite=False)
print(updated)  # 只有原来的 B、A 两行和 reading 列，读数为 20、10。
print(updated.shape, combined.shape)  # (2, 1)、(3, 2)。
print(first.loc["B", "reading"])  # nan：原始 first 未被修改。

   reading
B     20.0
A     10.0
(2, 1) (3, 2)
nan


## 8 缺失值与统计口径

sum、mean 等统计默认跳过缺失值；设置 skipna=False 则保留缺失对结果的影响。有效样本数量与总行数不同，要同时核对 count 和 size。

sum 的 min_count 默认是 0，全缺失或空输入的和可以是 0。如果任务要求至少一条有效观测，设置 min_count=1，使“没有观测”与“观测之和为零”能够区分。

In [12]:
values = pd.Series([2.0, np.nan, 4.0])
print(values.sum(), values.mean(), values.count(), values.size)  # 6、3、2、3。
print(values.sum(skipna=False), values.mean(skipna=False))  # 都为 nan。
missing = pd.Series([np.nan, np.nan], dtype="float64")
empty = pd.Series([], dtype="float64")
print(missing.sum(), missing.sum(min_count=1))  # 0.0、nan。
print(empty.sum(), empty.sum(min_count=1))  # 0.0、nan。
print(pd.Series([2, None], dtype="Int64").sum(min_count=2))  # <NA>，不足两项有效值。

6.0 3.0 2 3
nan nan
0.0 nan
0.0 nan
<NA>


## 9 缺失条件与筛选

可空布尔条件用于筛选时，NA 位置不被选中。这是筛选规则，不表示这个未知条件在逻辑上变成了 False。

下面一条读数缺失，无法判断它是否大于 10。若目标是“选取已确认大于 10 的观测”，直接筛选；若目标还包括待调查的未知记录，显式补 True，或把未知记录另存。

In [13]:
values = pd.Series([5, None, 20], index=["A", "B", "C"], dtype="Int64")
condition = values > 10
print(condition)  # False、<NA>、True，dtype 为 boolean。
print(values.loc[condition])  # 仅 C=20。
print(values.loc[condition.fillna(True)])  # B 的缺失和 C=20 都保留。
print(values.loc[condition.isna()])  # 单独保留 B，供调查。

A    False
B     <NA>
C     True
dtype: boolean
C    20
dtype: Int64
B    <NA>
C      20
dtype: Int64
B    <NA>
dtype: Int64


## 10 重复数据行与保留规则

duplicated 返回与行对应的布尔 Series，标记重复记录；drop_duplicates 返回去重结果。默认按所有列判断，忽略行索引。subset 可改成按指定业务键判断。

keep='first' 保留出现的第一条，keep='last' 保留最后一条，keep=False 删除重复组的所有行。出现顺序不等于最新时间；需要按时间选取时，应先明确排序规则。

In [14]:
records = pd.DataFrame({"device": ["A", "A", "B", "B"], "value": [10, 10, 20, 21]},
                       index=["r1", "r2", "r3", "r4"])
print(records.duplicated().tolist())  # [False, True, False, False]，仅 r2 与已有整行重复。
print(records.duplicated(subset=["device"], keep=False).tolist())  # 全 True，两组键都重复。
print(records.drop_duplicates())  # 按整行去重，保留 r1、r3、r4。
print(records.drop_duplicates(subset=["device"], keep="last"))  # 保留 r2、r4。
print(records.drop_duplicates(subset=["device"], keep=False).shape)  # (0, 2)。

[False, True, False, False]
[True, True, True, True]
   device  value
r1      A     10
r3      B     20
r4      B     21
   device  value
r2      A     10
r4      B     21
(0, 2)


重复行与重复标签是两件事。两行可以标签相同但数据不同，也可以标签不同但数据相同。

用 index.is_unique 和 index.duplicated 检查行标签。ignore_index=True 只给去重结果重新编号，不是检查业务键或解决冲突的替代方法。

In [15]:
records = pd.DataFrame({"value": [10, 20, 20]}, index=["A", "A", "B"])
print(records.index.is_unique)  # False：两个 A 标签。
print(records.index.duplicated(keep=False).tolist())  # [True, True, False]。
print(records.duplicated().tolist())  # [False, False, True]：第三行数值与第二行重复。
deduplicated = records.drop_duplicates(ignore_index=True)
print(deduplicated)  # 10、20 两条记录，重新编号 0、1。
print(records.index.tolist())  # 原索引仍是 A、A、B。

False
[True, True, False]
[False, False, True]
   value
0     10
1     20
['A', 'A', 'B']


## 11 选学：插值的条件
interpolate 根据已有观测估计缺失值，不能恢复未知的真实观测。默认 method='linear' 忽略索引间距，把行当作等距；method='index' 使用数值索引的实际距离。

下面索引表示位置，单位为米，观测值单位为摄氏度。位置 1 位于 0 和 4 之间的四分之一处，按位置线性插值得到 1；按行等距处理则得到 2。是否适合假设线性变化，由数据任务决定。

In [16]:
values = pd.Series([0.0, np.nan, 4.0], index=[0.0, 1.0, 4.0])
print(values.interpolate(method="linear").tolist())  # [0, 2, 4]：按行等距。
print(values.interpolate(method="index").tolist())  # [0, 1, 4]：按真实位置。
print(values.tolist())  # 原输入仍保留中间缺失。

[0.0, 2.0, 4.0]
[0.0, 1.0, 4.0]
[0.0, nan, 4.0]


limit 限制连续缺失的填充数量，limit_direction 控制处理方向，limit_area='inside' 只补两侧都有有效值的内部空缺。下面明确保留两端缺失，避免把边缘延续值当作有观测支持的插值。

本章使用 float64 的线性方法，不增加 SciPy。其他插值方法可能要求额外依赖和参数；整数、类别与文本也不能因为“有空缺”就直接套用连续数值插值。

In [17]:
values = pd.Series([np.nan, 0.0, np.nan, np.nan, 3.0, np.nan])
inside = values.interpolate(method="linear", limit=2, limit_direction="forward", limit_area="inside")
print(inside.tolist())  # [nan, 0, 1, 2, 3, nan]：两端仍缺失。
print(inside.isna().tolist())  # [True, False, False, False, False, True]。

[nan, 0.0, 1.0, 2.0, 3.0, nan]
[True, False, False, False, False, True]


## 本章小结

（1）缺失标记依赖 dtype，统一用 isna、notna 判断；空字符串和无穷值需要单独约定。

（2）删除、填充、替换与传播有不同含义。明确列规则、顺序和有效样本口径，再选择参数。

（3）combine_first 优先保留原表非缺失值，并返回标签并集；update 修改原对象且不扩展范围。

（4）重复记录和重复标签分别检查。按业务键去重要说明保留哪条，以及当前顺序的意义。

（5）插值是基于条件的估计，需区分行间距与实际坐标；未知值和补出的估计不应混为真实观测。

## 练习

（1）设备编号为必填项；备注缺失可写成“待补充”。清洗后保留原输入，检查行标签、行数、缺失和类型。不要因为一个非必填字段缺失就删除该行。

In [18]:
raw = pd.DataFrame({"device": ["A", None, "C"], "note": [None, "复核", "正常"]},
                   index=["r1", "r2", "r3"])
# 在此按列规则清洗并打印检查。
# 检查：只删除 r2；r1 备注补为待补充，原输入不变。

（2）先预测两次求和与筛选的结果，再运行。现在要求“没有有效观测时不能报告总量为零”，求和应增加哪个参数？说明含 NA 条件没有选中该行的原因。

In [19]:
values = pd.Series([None, None], dtype="Int64")
print(values.sum())
print(values.sum(min_count=1))
condition = pd.Series([True, pd.NA], dtype="boolean")
print(values.loc[condition])
# 在此解释两种总量口径及未知条件的筛选规则。

0
<NA>
0    <NA>
dtype: Int64


（3）主表中的读数优先，备表可补缺失。第一次要求合入新增设备，第二次要求严格保留原表范围。分别选择 combine_first 或 update，并解释参数和原表修改的区别。

In [20]:
primary = pd.DataFrame({"value": [1.0, np.nan]}, index=["A", "B"])
backup = pd.DataFrame({"value": [3.0, 2.0, 9.0]}, index=["C", "B", "A"])
# 在此两种要求分别从 primary 的未改动输入开始计算。
# 检查：A 始终为 1，B 补为 2；只有第一种要求包含 C，原 primary 应保留。

（4）数据已按录入先后排列，任务规定每个设备保留最后录入的一条。先列出全部重复键记录，再去重；解释为什么这与“仅移除完全相同的行”不同。

In [21]:
records = pd.DataFrame({"device": ["A", "B", "A", "C"], "value": [10, 20, 11, 30]})
# 在此查看重复键记录，按规则去重并重新编号。
# 检查：A 最终为 11；B、C 保留，三条记录的设备键唯一。

### 重点练习提示（第 3 题）

提示一：两次任务都优先保留主表值，区别在是否允许新增索引。

提示二：扩大范围用 combine_first；固定范围则在副本上 update，并禁止覆盖非缺失值。

### 参考解析（第 3 题）

primary.combine_first(backup) 返回 A=1.0、B=2.0、C=3.0，索引取两表并集，shape 为 (3, 1)。第二种要求先复制 primary，再 update(backup, overwrite=False)，只得到 A=1.0、B=2.0，shape 为 (2, 1)。两者的 value 均为 float64；primary 自身仍是 A=1.0、B 缺失。若给 update 保留默认 overwrite=True，备表的 A=9.0 会覆盖主表，违反本题优先级。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | pandas 3.0.6：[Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) 的 Values considered missing、NA semantics、Calculations、Filling missing data；[String migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的默认 str、NaN、PyArrow 后端；[isna](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) 的空串与无穷值说明；[replace](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html) 的按列映射；[dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) 的 subset、axis、how、thresh；[fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) 的按列填充；[ffill](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html)、[bfill](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.bfill.html) 的方向及 limit；[combine_first](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.combine_first.html)、[update](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.update.html) 的标签范围、优先级与修改行为；[sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) 的 skipna、min_count 与空输入；[Nullable Boolean](https://pandas.pydata.org/docs/user_guide/boolean.html#nullable-boolean) 的 Indexing with NA；[duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)、[drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) 的 subset、keep、索引忽略及重新编号；[Duplicate Labels](https://pandas.pydata.org/docs/user_guide/duplicates.html) 的 is_unique、Index.duplicated；[interpolate](https://pandas.pydata.org/docs/reference/api/pandas.Series.interpolate.html) 的 method、limit、limit_direction、limit_area 和额外依赖。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[missing_data](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/missing_data.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[boolean](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/boolean.rst)、[duplicates](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/duplicates.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |